# Run the daily update

This notebook refreshes prices, weather, and storage, scores every tournament contender, backfills matured actuals, and publishes the stitched production forecast. It calls the same `ScoringPipeline` as `natgas score` and the scheduled job. Already scored days are skipped, so it is safe to rerun.

## Setup

In [ ]:
# Import torch first: on Windows, loading numpy or mlflow before torch can raise WinError 1114.
import torch  # noqa: F401

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from natgas.config import load_settings
from natgas.forecasting.production.PublishedForecast import PublishedForecast
from natgas.forecasting.production.ScoringFlow import ScoringFlow
from natgas.forecasting.production.ScoringPipeline import ScoringPipeline
from natgas.forecasting.production.tournament.TournamentConfig import TournamentConfig
from natgas.forecasting.production.tournament.TournamentStore import TournamentStore
from natgas.forecasting.tuning.TuningConfig import TuningConfig

# The notebook can be opened from notebooks/, so move to the repo root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

%matplotlib inline

## Parameters

Set `refresh` to `False` to score from the data already loaded.

In [ ]:
refresh = True
config = "config/tuning.yaml"

## Run

In [ ]:
run = ScoringPipeline(
    load_settings(),
    TuningConfig.from_yaml(config),
    refresh=refresh,
    tournament=TournamentConfig.from_yaml(config),
).run()
summary = ScoringFlow.summary(run)

print(f"run {summary['run_id']} as of {summary['as_of_date']}")
print(f"rows loaded: {summary['loaded']}")
print(f"predictions saved: {summary['saved']}")
print(f"matured actuals: {summary['actuals']}")
print("event winners:")
for bucket, winner in summary["events"].items():
    print(f"  {bucket}: {winner}")
print(f"promotions: {summary['promoted'] or 'none'}")
print(f"production rows published: {summary['stitched']}")

## Results

The latest published production forecast (30 trading days) plotted against recent actual prices, plus the tournament standings and challenger trial status.

In [ ]:
settings = load_settings()
published = PublishedForecast(settings.db_url)
forecast = published.latest()
prices = published.prices()
store = TournamentStore(settings.db_url)
standings = store.latest("tournament_standings")
trials = store.latest("challenger_trials")

if forecast.empty:
    print("no published forecast yet, run the cell above first")
else:
    as_of = forecast["as_of_date"].iloc[0]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(pd.to_datetime(prices["date"]), prices["high"], label="actual")
    ax.plot(
        pd.to_datetime(forecast["target_date"]),
        forecast["forecast"],
        label="production forecast",
    )
    if forecast["lower"].notna().any():
        ax.fill_between(
            pd.to_datetime(forecast["target_date"]),
            forecast["lower"],
            forecast["upper"],
            alpha=0.2,
            label="interval",
        )
    ax.axvline(pd.to_datetime(as_of), color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"Published forecast as of {as_of}")
    ax.set_ylabel("NG=F daily high, $/MMBtu")
    ax.legend()
    fig.tight_layout()
    plt.show()

standings_cols = ["horizon_bucket", "rank", "stage", "prod_days", "blended_mae", "qualified"]
trials_cols = ["family", "trial_days", "required_days", "champion_mae", "challenger_mae", "status"]
if standings.empty:
    print("no tournament standings yet")
else:
    display(standings.sort_values(["horizon_bucket", "rank"])[standings_cols])
if trials.empty:
    print("no challenger trials yet")
else:
    display(trials[trials_cols])